# Client-level behavioural features (D10)

Revision_Plan_V03.md, Phase 3, D10. Reviewer 1's deepest criticism was that every
feature in the paper describes the *population* — the transaction category, the
city, the occupation — whereas fraud detection is about behaviour that is unusual
*for this cardholder*. This notebook adds seven features that describe each
transaction relative to the same card's own history.

**Every feature is causal**: it uses only the card's transactions strictly *before*
the current one, as a deployed system would. None of them uses `is_fraud`.

Input: `fraudTrain.csv` (for `cc_num`, timestamps, merchant coordinates, category)
and `fraudTrain_encoded.csv`. The two files are row-aligned — amount, fraud label
and timestamp agree row for row across all 1,296,675 transactions — so the new
columns are joined by position. Output: `fraudTrain_encoded_client.csv` = the 28
encoded predictors + the 7 client features + `is_fraud`.

Pipeline position: run after `MLProj01_fraudTrain_encoding_V24` and before the
model notebook.


In [1]:
import os
from pathlib import Path

# Run from the project directory, the one containing fraudTrain_encoded.csv.
path = str(Path.cwd()) + os.sep
if not (Path(path) / "fraudTrain_encoded.csv").exists():
    print("fraudTrain_encoded.csv not found: start Jupyter in the project directory")
os.chdir(path)
print("working directory:", path)


working directory: ./


In [2]:
raw = pd.read_csv(path + "fraudTrain.csv",
                  usecols=["cc_num", "unix_time", "trans_date_trans_time", "amt",
                           "category", "merch_lat", "merch_long", "lat", "long", "is_fraud"])
enc = pd.read_csv(path + "fraudTrain_encoded.csv")

# The join is by position, so check the alignment rather than assume it.
assert len(raw) == len(enc)
assert np.allclose(raw["amt"].values, enc["amt"].values)
assert (raw["is_fraud"].values == enc["is_fraud"].values).all()
_ts = pd.to_datetime(raw["trans_date_trans_time"])
assert (((_ts - _ts.min()).dt.total_seconds() // 3600).astype(int).values
        == enc["trans_time"].values).all()
print(f"{len(raw):,} rows, aligned; {raw['cc_num'].nunique()} cards")


1,296,675 rows, aligned; 983 cards


## Feature definitions

| Feature | Meaning |
|---|---|
| `card_amt_zscore` | how unusual this amount is for this card: (amount − mean of the card's previous amounts) / their standard deviation. 0 until the card has 5 prior transactions; clipped to ±10 |
| `card_hours_since_prev` | hours since the card's previous transaction, capped at 720 (30 days); 720 for a card's first transaction |
| `card_txn_1h`, `card_txn_24h`, `card_txn_7d` | number of the card's transactions in the preceding 1 hour, 24 hours and 7 days (not counting this one) |
| `card_dist_from_usual_km` | distance in km from this merchant to the centroid of the card's previous merchants; the cardholder's home for a first transaction |
| `card_new_category` | 1 if this is the card's first purchase in this category, else 0 |

A count of the card's prior transactions is deliberately **not** included: it rises
monotonically with time, so under the chronological split it would behave as a
disguised row index — the same problem that led to `trans_time` being dropped (B4).


In [3]:
MIN_HISTORY = 5          # prior transactions needed before a z-score is trusted
HOURS_CAP = 720          # 30 days

# Order each card's history in time. The raw file is not globally time-ordered;
# ties on unix_time are broken by original row order.
df = raw.copy()
df["_row"] = np.arange(len(df))
df = df.sort_values(["cc_num", "unix_time", "_row"], kind="mergesort").reset_index(drop=True)
g = df.groupby("cc_num", sort=False)

# --- amount relative to the card's own past ---------------------------------
k = g.cumcount()                                        # number of prior txns
s1 = g["amt"].cumsum() - df["amt"]                      # sum of prior amounts
s2 = (df["amt"] ** 2).groupby(df["cc_num"]).cumsum() - df["amt"] ** 2
mean_prev = s1 / k.replace(0, np.nan)
var_prev = (s2 - k * mean_prev ** 2) / (k - 1).replace(0, np.nan)
std_prev = np.sqrt(var_prev.clip(lower=0))
z = (df["amt"] - mean_prev) / std_prev.replace(0, np.nan)
df["card_amt_zscore"] = z.where(k >= MIN_HISTORY, 0.0).fillna(0.0).clip(-10, 10)

# --- time since the card's previous transaction ------------------------------
dt_h = g["unix_time"].diff() / 3600.0
df["card_hours_since_prev"] = dt_h.fillna(HOURS_CAP).clip(upper=HOURS_CAP)

# --- trailing counts ---------------------------------------------------------
def trailing_counts(times, window_s):
    # times are sorted within the card; for position i, count prior positions
    # with time in (t_i - window, t_i].
    j = np.searchsorted(times, times - window_s, side="right")
    return np.maximum(np.arange(len(times)) - j, 0)

for col, w in [("card_txn_1h", 3600), ("card_txn_24h", 86400), ("card_txn_7d", 7 * 86400)]:
    out = np.empty(len(df), dtype=np.int32)
    for idx in g.indices.values():
        out[idx] = trailing_counts(df["unix_time"].values[idx], w)
    df[col] = out

# --- distance from the card's usual merchant location ------------------------
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

c_lat = (g["merch_lat"].cumsum() - df["merch_lat"]) / k.replace(0, np.nan)
c_lon = (g["merch_long"].cumsum() - df["merch_long"]) / k.replace(0, np.nan)
c_lat, c_lon = c_lat.fillna(df["lat"]), c_lon.fillna(df["long"])   # first txn: home
df["card_dist_from_usual_km"] = haversine_km(c_lat, c_lon, df["merch_lat"], df["merch_long"])

# --- first purchase in this category -----------------------------------------
df["card_new_category"] = (df.groupby(["cc_num", "category"], sort=False).cumcount() == 0).astype(int)

CLIENT_FEATURES = ["card_amt_zscore", "card_hours_since_prev", "card_txn_1h", "card_txn_24h",
                   "card_txn_7d", "card_dist_from_usual_km", "card_new_category"]

# back to the original row order, to line up with the encoded file
df = df.sort_values("_row").reset_index(drop=True)
assert (df["_row"].values == np.arange(len(df))).all()
df[CLIENT_FEATURES].describe().T


,count,mean,std,min,25%,50%,75%,max
card_amt_zscore,1296675.0,0.001426,0.957970,-3.726692,-0.431734,-0.188287,0.110961,10.000000
card_hours_since_prev,1296675.0,9.562601,23.573551,0.000000,1.674444,4.613889,11.210833,720.000000
card_txn_1h,1296675.0,0.189271,0.461451,0.000000,0.000000,0.000000,0.000000,6.000000
card_txn_24h,1296675.0,3.882335,3.081123,0.000000,2.000000,3.000000,5.000000,35.000000
card_txn_7d,1296675.0,25.096393,13.889165,0.000000,15.000000,23.000000,33.000000,105.000000
card_dist_from_usual_km,1296675.0,76.272197,29.333366,0.181183,55.352030,78.276547,98.672703,250.501323
card_new_category,1296675.0,0.010094,0.099958,0.000000,0.000000,0.000000,0.000000,1.000000


## Causality check

Recompute every feature for a sample of cards by brute force — for each transaction,
filter the card's history to strictly earlier transactions and compute the feature
directly — and require an exact match with the vectorised version above. This is the
guard against a feature silently using the current or a future transaction.


In [4]:
rng = np.random.default_rng(1200)
sample_cards = rng.choice(raw["cc_num"].unique(), size=25, replace=False)
checked = 0
for card in sample_cards:
    rows = np.where(raw["cc_num"].values == card)[0]
    order = rows[np.lexsort((rows, raw["unix_time"].values[rows]))]   # time, then row
    for pos, r in enumerate(order):
        prev = order[:pos]
        t = raw["unix_time"].values[r]
        a_prev = raw["amt"].values[prev]
        # amount z-score
        if len(prev) >= MIN_HISTORY and a_prev.std(ddof=1) > 0:
            exp_z = np.clip((raw["amt"].values[r] - a_prev.mean()) / a_prev.std(ddof=1), -10, 10)
        else:
            exp_z = 0.0
        assert np.isclose(df.at[r, "card_amt_zscore"], exp_z, atol=1e-6), (card, pos, "zscore")
        # hours since previous
        exp_h = HOURS_CAP if not len(prev) else min((t - raw["unix_time"].values[prev[-1]]) / 3600, HOURS_CAP)
        assert np.isclose(df.at[r, "card_hours_since_prev"], exp_h), (card, pos, "hours")
        # trailing counts
        tp = raw["unix_time"].values[prev]
        for col, w in [("card_txn_1h", 3600), ("card_txn_24h", 86400), ("card_txn_7d", 7 * 86400)]:
            assert df.at[r, col] == int(((tp > t - w) & (tp <= t)).sum()), (card, pos, col)
        # distance from usual location
        if len(prev):
            clat, clon = raw["merch_lat"].values[prev].mean(), raw["merch_long"].values[prev].mean()
        else:
            clat, clon = raw["lat"].values[r], raw["long"].values[r]
        exp_d = haversine_km(clat, clon, raw["merch_lat"].values[r], raw["merch_long"].values[r])
        assert np.isclose(df.at[r, "card_dist_from_usual_km"], exp_d, rtol=1e-9, atol=1e-6), (card, pos, "dist")
        # new category
        exp_new = int(raw["category"].values[r] not in set(raw["category"].values[prev]))
        assert df.at[r, "card_new_category"] == exp_new, (card, pos, "category")
        checked += 1
print(f"causality check passed: {checked:,} transactions across {len(sample_cards)} cards, all 7 features")


causality check passed: 32,411 transactions across 25 cards, all 7 features


In [5]:
out = enc.copy()
for c in CLIENT_FEATURES:
    out.insert(out.columns.get_loc("is_fraud"), c, df[c].values)
out.to_csv(path + "fraudTrain_encoded_client.csv", index=False)
print("wrote fraudTrain_encoded_client.csv:", out.shape)

# how the features differ between fraudulent and genuine transactions
out.groupby("is_fraud")[CLIENT_FEATURES].median().T


wrote fraudTrain_encoded_client.csv: (1296675, 36)


is_fraud,0,1
card_amt_zscore,-0.190093,2.38627
card_hours_since_prev,4.630278,1.40875
card_txn_1h,0.000000,0.00000
card_txn_24h,3.000000,4.00000
card_txn_7d,23.000000,17.00000
card_dist_from_usual_km,78.276650,78.22217
card_new_category,0.000000,0.00000
